# 04_SES_Feature_Engineering_Final_v2

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
pd.set_option('display.max_columns',None)


In [2]:
job_skills = pd.read_csv('job_skills.csv')
job_skills_map = pd.read_csv('job_skills_1.csv')
skills_map = pd.read_csv('skills_1.csv')
job_industries = pd.read_csv('job_industries.csv')
linkedin = pd.read_csv('linkedin_job_postings.csv')
salaries = pd.read_csv('salaries.csv')
stack = pd.read_csv('survey_results_public.csv',low_memory=False)


In [3]:
TECH_SKILLS=[
'python','sql','javascript','typescript','java','c++','c#','go','rust',
'html/css','react','next.js','node.js','docker','kubernetes',
'amazon web services (aws)','aws','microsoft azure','azure',
'google cloud','gcp','mongodb','postgresql','mysql','redis',
'tensorflow','pytorch','scikit-learn','spark','power bi','tableau'
]


In [12]:
all_skills=[]
for row in job_skills['job_skills'].dropna():
    all_skills.extend([x.strip().lower() for x in str(row).split(',')])

demand_df=pd.Series(all_skills).value_counts().reset_index()
demand_df.columns=['skill','linkedin_demand']
demand_df=demand_df[demand_df['skill'].isin(TECH_SKILLS)]
demand_df.head(100)


,skill,linkedin_demand
83,python,25168
97,sql,22035
179,java,13891
193,aws,12738
261,javascript,9451
360,azure,7264
374,tableau,7027
395,kubernetes,6711
396,c++,6654
421,docker,6254


In [5]:
skill_lookup=(job_skills_map
              .merge(skills_map,on='skill_abr',how='left'))
skill_lookup['skill_name']=skill_lookup['skill_name'].astype(str).str.lower()

industry_adoption=(skill_lookup
                   .merge(job_industries,on='job_id',how='left')
                   .groupby('skill_name')['industry_id']
                   .nunique()
                   .reset_index())

industry_adoption.columns=['skill','industry_adoption']
industry_adoption=industry_adoption[industry_adoption['skill'].isin(TECH_SKILLS)]


In [6]:
salary_feature=(salaries
    .merge(job_skills_map,on='job_id')
    .merge(skills_map,on='skill_abr'))

salary_feature['skill_name']=salary_feature['skill_name'].str.lower()

salary_feature['salary_premium']=(
    salary_feature['max_salary'].fillna(0)+
    salary_feature['min_salary'].fillna(0))/2

salary_feature=(salary_feature
                .groupby('skill_name')['salary_premium']
                .mean()
                .reset_index())

salary_feature.columns=['skill','salary_premium']
salary_feature=salary_feature[salary_feature['skill'].isin(TECH_SKILLS)]


In [7]:
def extract_counts(series):
    c=Counter()
    for row in series.dropna():
        for item in str(row).split(';'):
            c[item.strip().lower()]+=1
    return c


In [8]:
usage_cols=['LanguageHaveWorkedWith','DatabaseHaveWorkedWith',
'PlatformHaveWorkedWith','WebframeHaveWorkedWith',
'ToolsTechHaveWorkedWith','MiscTechHaveWorkedWith']

current=Counter()
for col in usage_cols:
    current.update(extract_counts(stack[col]))

current_usage_df=pd.DataFrame(current.items(),columns=['skill','current_usage'])
current_usage_df=current_usage_df[current_usage_df['skill'].isin(TECH_SKILLS)]


In [9]:
future_cols=['LanguageWantToWorkWith','DatabaseWantToWorkWith',
'PlatformWantToWorkWith','WebframeWantToWorkWith',
'ToolsTechWantToWorkWith','MiscTechWantToWorkWith']

future=Counter()
for col in future_cols:
    future.update(extract_counts(stack[col]))

future_interest_df=pd.DataFrame(future.items(),columns=['skill','future_interest'])
future_interest_df=future_interest_df[future_interest_df['skill'].isin(TECH_SKILLS)]


In [10]:
master=demand_df.merge(industry_adoption,on='skill',how='outer')
master=master.merge(salary_feature,on='skill',how='outer')
master=master.merge(current_usage_df,on='skill',how='outer')
master=master.merge(future_interest_df,on='skill',how='outer')

master['geographic_spread']=1
master.fillna(0,inplace=True)

features=['linkedin_demand','industry_adoption',
          'salary_premium','current_usage',
          'future_interest','geographic_spread']

master[features]=MinMaxScaler().fit_transform(master[features])

master['future_score']=(
0.30*master['future_interest']+
0.25*master['linkedin_demand']+
0.15*master['current_usage']+
0.15*master['industry_adoption']+
0.10*master['geographic_spread']+
0.05*master['salary_premium']
)

master.sort_values('future_score',ascending=False).head(50)


,skill,linkedin_demand,industry_adoption,salary_premium,current_usage,future_interest,geographic_spread,future_score
20,python,1.000000,0.0,0.0,0.819348,0.953735,0.0,0.659023
27,sql,0.875517,0.0,0.0,0.818361,0.852943,0.0,0.597516
11,javascript,0.375517,0.0,0.0,1.000000,0.905262,0.0,0.515458
5,docker,0.248490,0.0,0.0,0.779340,1.000000,0.0,0.479023
18,postgresql,0.064129,0.0,0.0,0.681105,0.914058,0.0,0.392416
9,html/css,0.032661,0.0,0.0,0.848608,0.789011,0.0,0.372160
30,typescript,0.127106,0.0,0.0,0.617465,0.770657,0.0,0.355593
10,java,0.551931,0.0,0.0,0.486477,0.406214,0.0,0.332819
0,amazon web services (aws),0.014423,0.0,0.0,0.591886,0.686924,0.0,0.298466
22,react,0.144628,0.0,0.0,0.511229,0.586551,0.0,0.288807


In [11]:
master.to_csv('master_skill_dataset_v2.csv',index=False)
print('Saved: master_skill_dataset_v2.csv')


Saved: master_skill_dataset_v2.csv
